# 03 — Validación de Datasets Procesados

**Fase 1 — Semana 2**  
Objetivo: verificar que los datasets generados en `02_preprocessing.ipynb` son correctos
antes de entrenar los modelos generativos.

Checks:
1. Shapes y tipos
2. Variable objetivo (mortalidad)
3. Missingness post-imputación
4. Rangos clínicos de variables clave
5. Correlaciones entre variables
6. Coherencia tabular ↔ timeseries (estancias comunes)
7. Rango de normalización del array de series temporales

## 0. Imports y configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

ROOT      = Path("..")
PROCESSED = ROOT / "data" / "processed"
REPORTS   = ROOT / "reports"
REPORTS.mkdir(exist_ok=True)

np.random.seed(42)
sns.set_theme(style="whitegrid", palette="muted")

print("Rutas:")
print(f"  PROCESSED: {PROCESSED.resolve()}  {'OK' if PROCESSED.exists() else 'NO EXISTE'}")

## 1. Carga de datasets

In [ ]:
tab  = pd.read_parquet(PROCESSED / "tabular_48h.parquet")
ts   = np.load(PROCESSED / "timeseries_48h.npy")
meta = pd.read_parquet(PROCESSED / "timeseries_48h_meta.parquet")
norm = pd.read_csv(PROCESSED / "timeseries_norm_params.csv")

print("=== TABULAR ===")
print(f"  Shape:    {tab.shape}")
print(f"  Dtypes:   {tab.dtypes.value_counts().to_dict()}")
print()
print("=== TIMESERIES ===")
print(f"  Array:    {ts.shape}  (estancias × horas × vitales)")
print(f"  Meta:     {meta.shape}")
print(f"  Norm:     {norm.shape}")

## 2. Variable objetivo — mortalidad

In [ ]:
# Tabular
mort_tab  = tab["hospital_expire_flag"].mean()
mort_meta = meta["hospital_expire_flag"].mean()
n_tab     = len(tab)
n_ts      = len(meta)

print(f"Tabular  — estancias: {n_tab:,}  mortalidad: {mort_tab:.3f} ({mort_tab*100:.1f}%)")
print(f"Timeseries — estancias: {n_ts:,}  mortalidad: {mort_meta:.3f} ({mort_meta*100:.1f}%)")
print()

# Diferencia de estancias
diff = n_tab - n_ts
ids_tab = set(tab["icustay_id"])
ids_ts  = set(meta["icustay_id"])
solo_tab = ids_tab - ids_ts
print(f"Estancias solo en tabular (sin timeseries): {len(solo_tab)}")
if len(solo_tab) > 0:
    excluidas = tab[tab["icustay_id"].isin(solo_tab)][["icustay_id", "los", "hospital_expire_flag"]]
    print(f"  LOS medio de excluidas: {excluidas['los'].mean():.2f} días")
    print(f"  Mortalidad excluidas:   {excluidas['hospital_expire_flag'].mean():.3f}")
    print("  (Probable causa: vitales insuficientes para cubrir 48h completas)")

## 3. Missingness post-imputación

In [ ]:
missing = tab.isnull().mean().mul(100).sort_values(ascending=False)
missing_nz = missing[missing > 0]

print(f"Columnas con missingness > 0: {len(missing_nz)} / {len(tab.columns)}")
if len(missing_nz) > 0:
    print(missing_nz.to_string())
else:
    print("  Missingness global = 0.0% — imputación completa OK")

# NaNs en timeseries
nan_ts = np.isnan(ts).sum()
print(f"\nNaNs en array timeseries: {nan_ts}  {'OK' if nan_ts == 0 else 'PROBLEMA'}")

## 4. Rangos clínicos de variables clave

Valores de referencia UCI:
- Heart rate: 30–200 bpm
- SBP: 40–250 mmHg
- SpO2: 50–100 %
- Temperatura: 25–42 °C
- Resp rate: 5–60 rpm
- GCS total: 3–15
- Creatinina: 0–20 mg/dL
- Lactato: 0–30 mmol/L

In [ ]:
CLINICAL_RANGES = {
    "heart_rate_mean": (30, 200),
    "sbp_mean":        (40, 250),
    "spo2_mean":       (50, 100),
    "temp_c_mean":     (25, 42),
    "resp_rate_mean":  (5, 60),
    "gcs_total_mean":  (3, 15),
    "creatinine_mean": (0, 20),
    "lactate_mean":    (0, 30),
    "glucose_mean":    (20, 1000),
    "ph_arterial_mean":(6.5, 7.8),
}

print(f"{'Variable':<25} {'Min':>8} {'Mean':>8} {'Max':>8} {'Rango ref':>15} {'Outliers':>10}")
print("-" * 80)
issues = []
for col, (lo, hi) in CLINICAL_RANGES.items():
    if col not in tab.columns:
        print(f"{col:<25}  NO ENCONTRADA")
        continue
    s = tab[col].dropna()
    out_of_range = ((s < lo) | (s > hi)).sum()
    pct = out_of_range / len(s) * 100
    flag = "REVISAR" if pct > 1 else ""
    print(f"{col:<25} {s.min():>8.2f} {s.mean():>8.2f} {s.max():>8.2f}  [{lo:>4}, {hi:>4}]  {out_of_range:>6} ({pct:.2f}%) {flag}")
    if pct > 1:
        issues.append(col)

print()
if issues:
    print(f"Variables con >1% outliers clínicos: {issues}")
else:
    print("Todos los rangos clínicos son correctos.")

In [ ]:
# Distribuciones de vitales clave por mortalidad
vitals_plot = ["heart_rate_mean", "sbp_mean", "spo2_mean", "gcs_total_mean",
               "resp_rate_mean", "lactate_mean"]
vitals_labels = ["FC (bpm)", "PAS (mmHg)", "SpO2 (%)", "GCS total",
                 "FR (rpm)", "Lactato (mmol/L)"]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for ax, col, label in zip(axes, vitals_plot, vitals_labels):
    if col not in tab.columns:
        ax.set_visible(False)
        continue
    for flag, color, lbl in [(0, "steelblue", "Superviviente"), (1, "tomato", "Fallecido")]:
        subset = tab[tab["hospital_expire_flag"] == flag][col].dropna()
        ax.hist(subset, bins=50, alpha=0.5, color=color, label=lbl, density=True)
    ax.set_title(label)
    ax.legend(fontsize=8)

fig.suptitle("Distribución de vitales por mortalidad", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(REPORTS / "validation_vitals_by_mortality.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Correlaciones entre variables

In [ ]:
# Correlación entre variables numéricas clave (medias de vitales + labs + target)
CORR_COLS = [
    "hospital_expire_flag", "age", "los",
    "heart_rate_mean", "sbp_mean", "spo2_mean", "resp_rate_mean", "gcs_total_mean",
    "creatinine_mean", "lactate_mean", "bun_mean", "ph_arterial_mean",
    "wbc_mean", "hemoglobin_mean", "platelet_mean"
]
corr_cols = [c for c in CORR_COLS if c in tab.columns]
corr = tab[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, ax=ax, annot_kws={"size": 8})
ax.set_title("Correlación entre variables clave (tabular_48h)", fontsize=12)
plt.tight_layout()
plt.savefig(REPORTS / "validation_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

# Correlaciones con mortalidad
print("Correlaciones con hospital_expire_flag:")
print(corr["hospital_expire_flag"].drop("hospital_expire_flag").sort_values())

## 6. Coherencia tabular ↔ timeseries

In [ ]:
# Verificar que mortalidad es consistente en las estancias comunes
common_ids = ids_tab & ids_ts
tab_common  = tab[tab["icustay_id"].isin(common_ids)].set_index("icustay_id")
meta_common = meta[meta["icustay_id"].isin(common_ids)].set_index("icustay_id")

merged = tab_common[["hospital_expire_flag"]].join(
    meta_common[["hospital_expire_flag"]], lsuffix="_tab", rsuffix="_meta"
)
discrepancias = (merged["hospital_expire_flag_tab"] != merged["hospital_expire_flag_meta"]).sum()
print(f"Estancias comunes: {len(common_ids):,}")
print(f"Discrepancias en hospital_expire_flag: {discrepancias}  {'OK' if discrepancias == 0 else 'PROBLEMA'}")

# Verificar que edad es consistente
if "age" in tab_common.columns and "age" in meta_common.columns:
    age_merged = tab_common[["age"]].join(meta_common[["age"]], lsuffix="_tab", rsuffix="_meta")
    age_diff   = (age_merged["age_tab"] - age_merged["age_meta"]).abs()
    print(f"Discrepancias en age (>0.1): {(age_diff > 0.1).sum()}  {'OK' if (age_diff > 0.1).sum() == 0 else 'REVISAR'}")

## 7. Rango de normalización del array timeseries

In [ ]:
# El array debe estar en [0, 1] tras la normalización Min-Max
ts_min = float(ts.min())
ts_max = float(ts.max())
print(f"Array timeseries — min: {ts_min:.4f}  max: {ts_max:.4f}")
print(f"Rango [0, 1]: {'OK' if ts_min >= 0 and ts_max <= 1 else 'FUERA DE RANGO'}")

# Por vital
vital_names = norm["vital"].tolist()
print(f"\nVitales en timeseries ({len(vital_names)}): {vital_names}")
print(f"\n{'Vital':<15} {'min':>8} {'max':>8} {'media':>8} {'% ceros':>10}")
print("-" * 55)
for i, v in enumerate(vital_names):
    canal = ts[:, :, i]
    pct_cero = (canal == 0).mean() * 100
    print(f"{v:<15} {canal.min():>8.4f} {canal.max():>8.4f} {canal.mean():>8.4f} {pct_cero:>9.1f}%")

In [ ]:
# Evolución temporal media de cada vital (supervivientes vs fallecidos)
mort_flags = meta["hospital_expire_flag"].values

fig, axes = plt.subplots(2, 5, figsize=(18, 7), sharey=False)
axes = axes.flatten()

for i, v in enumerate(vital_names):
    if i >= len(axes):
        break
    ax = axes[i]
    for flag, color, lbl in [(0, "steelblue", "Superviviente"), (1, "tomato", "Fallecido")]:
        subset = ts[mort_flags == flag, :, i]
        mean   = subset.mean(axis=0)
        std    = subset.std(axis=0)
        ax.plot(range(48), mean, color=color, label=lbl, linewidth=1.5)
        ax.fill_between(range(48), mean - std, mean + std, color=color, alpha=0.15)
    ax.set_title(v, fontsize=9)
    ax.set_xlabel("Hora", fontsize=7)
    if i == 0:
        ax.legend(fontsize=7)

# Ocultar ejes sobrantes
for j in range(len(vital_names), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Evolución temporal media (normalizada) por vital y mortalidad", fontsize=12)
plt.tight_layout()
plt.savefig(REPORTS / "validation_timeseries_mortality.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Resumen de validación

In [ ]:
print("=" * 55)
print("  RESUMEN DE VALIDACIÓN")
print("=" * 55)
checks = [
    ("Tabular shape",            f"{tab.shape}"),
    ("Timeseries shape",         f"{ts.shape}"),
    ("Mortalidad tabular",       f"{mort_tab*100:.1f}%"),
    ("Mortalidad timeseries",    f"{mort_meta*100:.1f}%"),
    ("Missingness tabular",      f"{tab.isnull().mean().mean()*100:.2f}%"),
    ("NaNs timeseries",          str(nan_ts)),
    ("Rango TS [0,1]",           "OK" if ts_min >= 0 and ts_max <= 1 else "REVISAR"),
    ("Coherencia target",        "OK" if discrepancias == 0 else f"{discrepancias} discrepancias"),
    ("Outliers clínicos",        "OK" if not issues else str(issues)),
]
for name, val in checks:
    print(f"  {name:<30} {val}")
print("=" * 55)
print("Datasets listos para Fase 2 — modelos generativos.")